# Single-Market Drift Walkthrough

Thi
First concrete look at the project's headline question — how much does a Kalshi MLB market's price drift between pre-game trading and live (in-game) trading? This notebook works with **N=1 market**: the moneyline-spread `DET2` contract on the May 10 2026 Detroit Tigers @ Kansas City Royals game (YES resolves true if DET wins by 2+ runs). Market ID: `KXMLBSPREAD-26MAY101920DETKC-DET2`.

The goal here is is to:

1. Explore the data shape and analytical primitives for one market in depth, so we can be confident in our approach when we scale to many markets.
2. Demonstrate the data shape end-to-end (market metadata, orderbook snapshot, trade history) for one real example.
3. Establish the analytical primitives (price metrics, time alignment, drift definition) we'll reuse when we scale to many markets.
4. Cache the data to parquet so this notebook renders cleanly on GitHub even when the Snowflake warehouse is paused.

### Prerequisites

This notebook reads from the **dbt staging/marts** (`fct_markets`, `fct_market_orderbooks`, `stg_kalshi_market_trades`), not from `RAW`. If you've just scraped fresh data for this market, run `dbt run` first so the marts pick up the new rows:

```bash
dbt run --project-dir dbt --profiles-dir dbt
```

The first cell run will hit Snowflake once and write parquet to `analysis/data/single_market_KXMLBSPREAD-26MAY101920DETKC-DET2/`. Subsequent re-runs load from that cache and don't need a warm warehouse.

In [31]:
import json
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import find_dotenv, load_dotenv

from snow_py.connection import SnowflakeManager
from snow_py.timezone import utc_to_eastern

# Walk up from the notebook's cwd to find the repo root .env
load_dotenv(find_dotenv())

MARKET_TICKER = "KXMLBSPREAD-26MAY101920DETKC-DET2"
DATA_DIR = Path("data") / f"single_market_{MARKET_TICKER}"

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


def cached_query(query: str, name: str, snowflake: SnowflakeManager | None = None) -> pd.DataFrame:
    """Load a Snowflake query result from parquet cache if present, else query and cache.

    Keeps every notebook re-executable without warehouse access as long as the cache
    is on disk — the rendered cell output in the .ipynb is what ships to GitHub.
    """
    cache_path = DATA_DIR / f"{name}.parquet"
    if cache_path.exists():
        print(f"  loaded {name} from cache ({cache_path})")
        return pd.read_parquet(cache_path)
    if snowflake is None:
        raise RuntimeError(
            f"Cache miss for {name} and no Snowflake connection provided. "
            "Pass a SnowflakeManager to populate the cache."
        )
    print(f"  cache miss for {name}; querying Snowflake...")
    rows = snowflake.execute(query)
    df = pd.DataFrame(rows)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df.to_parquet(cache_path)
    print(f"  wrote {len(df)} rows to {cache_path}")
    return df

In [32]:
# Only open a Snowflake connection if we don't already have all three caches.
need_snowflake = not all(
    (DATA_DIR / f"{name}.parquet").exists()
    for name in ("market", "orderbook", "trades")
)

# Connect to PROD; use fully-qualified table names below so we don't depend on
# the session default schema. Per dbt/macros/generate_schema_name.sql, staging
# models materialize in PROD.STAGE and mart models materialize in PROD.KALSHI.
snowflake = SnowflakeManager("PROD", "STAGE") if need_snowflake else None

market_df = cached_query(
    f"""
    select *
    from PROD.KALSHI.fct_markets
    where market_ticker = '{MARKET_TICKER}'
    """,
    "market",
    snowflake,
)

orderbook_df = cached_query(
    f"""
    select *
    from PROD.KALSHI.fct_market_orderbooks
    where market_ticker = '{MARKET_TICKER}'
    """,
    "orderbook",
    snowflake,
)

trades_df = cached_query(
    f"""
    select *
    from PROD.STAGE.stg_kalshi_market_trades
    where market_ticker = '{MARKET_TICKER}'
    order by trade_time
    """,
    "trades",
    snowflake,
)

if snowflake is not None:
    snowflake.close()

# stg_kalshi_market_trades passes the numeric Kalshi columns straight through
# from a VARIANT source, so the Snowflake Python driver returns them as Python
# strings (e.g. "0.05"). Cast here so plots and stats work; this is a workaround
# for a dbt-side gap — the staging model should `cast(... as number)` to fix it
# for every downstream consumer (logged as a follow-up).
for col in ("yes_price_dollars", "no_price_dollars", "count_fp"):
    if col in trades_df.columns:
        trades_df[col] = pd.to_numeric(trades_df[col], errors="coerce")

# The staging model strips tz offsets via try_to_timestamp_ntz, so trade_time
# arrives as naive-UTC. Convert to tz-aware Eastern up-front so every downstream
# cell (plots, drift cuts, prints) sees viewer-friendly local time and DST is
# handled correctly.
trades_df["trade_time"] = utc_to_eastern(pd.to_datetime(trades_df["trade_time"]))

print()
print(f"market    : {len(market_df)} row(s)")
print(f"orderbook : {len(orderbook_df)} row(s)")
print(f"trades    : {len(trades_df)} rows")

INFO:snowflake.connector.connection:Snowflake Connector for Python Version: 4.3.0, Python Version: 3.13.12, Platform: Windows-11-10.0.26200-SP0
INFO:snowflake.connector.connection:Connecting to GLOBAL Snowflake domain


  cache miss for market; querying Snowflake...
  wrote 1 rows to data\single_market_KXMLBSPREAD-26MAY101920DETKC-DET2\market.parquet
  cache miss for orderbook; querying Snowflake...
  wrote 1 rows to data\single_market_KXMLBSPREAD-26MAY101920DETKC-DET2\orderbook.parquet
  cache miss for trades; querying Snowflake...
  wrote 862 rows to data\single_market_KXMLBSPREAD-26MAY101920DETKC-DET2\trades.parquet

market    : 1 row(s)
orderbook : 1 row(s)
trades    : 862 rows


## What this market is

The `fct_markets` row tells us what the contract resolves on, when trading opened and closed, and the latest price snapshot at scrape time. A few Kalshi field-name conventions worth keeping in mind:

- **`_dollars`** suffix — the value is denominated in dollars (e.g. `last_price_dollars = 0.42` means the last trade cleared at 42¢).
- **`_fp`** suffix — Kalshi's "fixed-point" convention; the value is an integer count of contracts with no decimal scaling. `volume_fp` is the lifetime contract count traded.
- **`yes_*` / `no_*`** — two sides of the binary market. YES pays $1 if the market resolves true, NO pays $1 if it resolves false. Prices sum to ~$1 (modulo the bid-ask spread).

In [33]:
assert len(market_df) == 1, f"Expected exactly one market row for {MARKET_TICKER}, got {len(market_df)}"
mkt = market_df.iloc[0]

print(f"Market : {mkt['market_ticker']}")
print(f"Event  : {mkt['event_title']} ({mkt['event_ticker']})")
print(f"Series : {mkt['series_ticker']}")
print(f"Type   : {mkt['market_type']}")
print(f"Status : {mkt['market_status']}")
print()
print("Trading window (Eastern):")
print(f"  opened at : {utc_to_eastern(mkt['open_at']):%Y-%m-%d %H:%M %Z}")
print(f"  closed at : {utc_to_eastern(mkt['close_at']):%Y-%m-%d %H:%M %Z}")
print()
print("Latest snapshot:")
print(f"  last price        : ${mkt['last_price_dollars']}")
print(f"  yes bid / ask     : ${mkt['yes_bid_dollars']} / ${mkt['yes_ask_dollars']}")
print(f"  no  bid / ask     : ${mkt['no_bid_dollars']} / ${mkt['no_ask_dollars']}")
print(f"  liquidity         : ${mkt['liquidity_dollars']}")
print(f"  volume (lifetime) : {mkt['volume_fp']}")
print()
print("Market title :", mkt["market_title"])
print("Yes resolves :", mkt["yes_subtitle"])
print("No  resolves :", mkt["no_subtitle"])

Market : KXMLBSPREAD-26MAY101920DETKC-DET2
Event  : Detroit vs Kansas City: Spread (KXMLBSPREAD-26MAY101920DETKC)
Series : KXMLBSPREAD
Type   : binary
Status : finalized

Trading window (Eastern):
  opened at : 2026-05-10 01:20 EDT
  closed at : 2026-05-10 22:24 EDT

Latest snapshot:
  last price        : $0.9900
  yes bid / ask     : $0.9900 / $1.0000
  no  bid / ask     : $0.0000 / $0.0100
  liquidity         : $0.0000
  volume (lifetime) : 82270.800000

Market title : Detroit wins by over 1.5 runs?
Yes resolves : Detroit wins by over 1.5 runs
No  resolves : Detroit wins by over 1.5 runs


## Trade timeline

How did YES price move between `open_at` and `close_at`? Plotting every executed trade with its timestamp. The vertical line marks the encoded first pitch — `26MAY101920DETKC` reads as **May 10 2026, 19:20 ET** (Kalshi's MLB event tickers use Eastern time).

All timestamps in this notebook were converted from naive-UTC (the dbt staging output) to tz-aware Eastern via `snow_py.timezone.utc_to_eastern`, so the chart x-axis and the drift cut below both read in ET without any mental conversion.

Color by `taker_side` to show which side was lifting the offer at each price point:

- **`yes`** — someone *bought* YES at the ask. Bullish on the outcome.
- **`no`** — someone *bought* NO at the ask, which is equivalent to *selling* YES. Bearish on the outcome.

(YES and NO prices on the same trade sum to ~$1, so plotting only YES gives the full picture.)

In [ ]:
import datetime as dt
from zoneinfo import ZoneInfo

from matplotlib.ticker import FormatStrFormatter, MaxNLocator

EASTERN = ZoneInfo("America/New_York")

# First-pitch is encoded in the event ticker as 19:20 ET; declare it tz-aware so
# it's directly comparable to trade_time (also tz-aware ET after pull-data).
GAME_START = dt.datetime(2026, 5, 10, 19, 20, tzinfo=EASTERN)

trades_df = trades_df.sort_values("trade_time").reset_index(drop=True)

# Sanity-check the boundary against the trading window and observed trade range.
print(f"Market open  : {utc_to_eastern(mkt['open_at']):%Y-%m-%d %H:%M %Z}")
print(f"Market close : {utc_to_eastern(mkt['close_at']):%Y-%m-%d %H:%M %Z}")
print(f"Trade range  : {trades_df['trade_time'].min():%Y-%m-%d %H:%M %Z} -> {trades_df['trade_time'].max():%Y-%m-%d %H:%M %Z}")
print(f"GAME_START   : {GAME_START:%Y-%m-%d %H:%M %Z}")
print(f"YES price    : ${trades_df['yes_price_dollars'].min():.3f} -> ${trades_df['yes_price_dollars'].max():.3f}")

fig, ax = plt.subplots(figsize=(11, 5))

for side, color, label in [
    ("yes", "tab:green", "taker bought YES"),
    ("no", "tab:red", "taker bought NO (sold YES)"),
]:
    subset = trades_df[trades_df["taker_side"] == side]
    ax.scatter(
        subset["trade_time"],
        subset["yes_price_dollars"],
        s=10,
        alpha=0.5,
        c=color,
        label=label,
    )

ax.axvline(GAME_START, color="black", linewidth=1.2, linestyle="--", label="game start (19:20 ET)")
ax.set_xlabel("trade time (ET)")
ax.set_ylabel("YES price")

# Cap to the [0, 1] payout range so a reader sees the price band relative to the
# two binary outcomes; cap further down only if the range is so narrow that this
# obscures the structure of the moves.
ax.set_ylim(0, 1)
ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax.yaxis.set_major_formatter(FormatStrFormatter("$%.2f"))

ax.set_title(f"{MARKET_TICKER} — YES price by trade")
ax.legend(loc="best")

# Matplotlib's date locator + formatter default to UTC even when the underlying
# data is tz-aware. Pass the Eastern tz to both so tick positions and labels
# read in ET (matching the rest of the notebook).
ax.xaxis.set_major_locator(mdates.AutoDateLocator(tz=EASTERN))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M", tz=EASTERN))

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Orderbook snapshot

Unlike trades, the orderbook is a single point-in-time picture of resting orders — a snapshot taken at scrape time, not a time series. The Snowflake `orderbook` column is a JSON `VARIANT` keyed by side:

- `yes_dollars` — a list of `[price, size]` pairs for resting **bids on YES** (someone willing to pay that price for YES).
- `no_dollars` — same for the NO side.

### A gotcha worth surfacing

**Once a market resolves, Kalshi removes all resting orders.** That means an orderbook snapshot for a `finalized` (or `settled`) market will come back empty even though the chain of `RAW_MARKET_ORDERBOOKS → STG_KALSHI_MARKET_ORDERBOOKS → FCT_MARKET_ORDERBOOKS` is doing exactly the right thing. This market (`market_status = finalized`) is in that state — both sides correctly show 0 price levels.

For meaningful orderbook visualizations we need to scrape a market that's still **open** at scrape time and re-run this notebook against that ticker. Logging that as a follow-up; the drift analysis below is unaffected because it works off the full trade history, which is preserved post-settlement.

In [35]:
assert len(orderbook_df) == 1, f"Expected exactly one orderbook row for {MARKET_TICKER}, got {len(orderbook_df)}"

# The orderbook column round-trips through Snowflake VARIANT -> JSON string.
orderbook_raw = orderbook_df.iloc[0]["orderbook"]
orderbook = json.loads(orderbook_raw) if isinstance(orderbook_raw, str) else orderbook_raw

# Diagnostic: surface what the dbt model computed and what the payload contains
# so an empty orderbook is debuggable rather than mysterious.
ob_row = orderbook_df.iloc[0]
print(f"market_status            : {ob_row['market_status']}")
print(f"market_title             : {ob_row['market_title']}")
print(f"Top-level orderbook keys : {list(orderbook.keys())}")
print(f"dbt yes_level_count      : {ob_row['yes_level_count']}")
print(f"dbt no_level_count       : {ob_row['no_level_count']}")
print()

yes_levels = pd.DataFrame(orderbook.get("yes_dollars") or [], columns=["price", "size"]).astype(float)
no_levels = pd.DataFrame(orderbook.get("no_dollars") or [], columns=["price", "size"]).astype(float)

if yes_levels.empty and no_levels.empty:
    print(
        "Orderbook is empty. If market_status is 'finalized' or 'settled' this is\n"
        "expected — Kalshi removes resting orders at settlement. Scrape an open\n"
        "market and re-run this notebook against that ticker for a populated chart."
    )
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    if not yes_levels.empty:
        ax.bar(yes_levels["price"], yes_levels["size"], width=0.005, color="tab:green", alpha=0.7, label="YES bids")
    if not no_levels.empty:
        ax.bar(no_levels["price"], no_levels["size"], width=0.005, color="tab:red", alpha=0.7, label="NO bids")
    ax.set_xlim(0, 1)
    ax.set_xlabel("price ($)")
    ax.set_ylabel("contracts resting")
    ax.set_title(f"{MARKET_TICKER} — orderbook snapshot at scrape time")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"YES side: {len(yes_levels)} price levels totalling {int(yes_levels['size'].sum())} contracts")
    print(f"NO  side: {len(no_levels)} price levels totalling {int(no_levels['size'].sum())} contracts")

market_status            : finalized
market_title             : Detroit wins by over 1.5 runs?
Top-level orderbook keys : ['no_dollars', 'yes_dollars']
dbt yes_level_count      : 0
dbt no_level_count       : 0

Orderbook is empty. If market_status is 'finalized' or 'settled' this is
expected — Kalshi removes resting orders at settlement. Scrape an open
market and re-run this notebook against that ticker for a populated chart.


## First-pass drift metric

Split the trades into **pre-game** and **live** windows using the encoded game start, then compare basic price statistics within each. The point isn't to draw a final conclusion from N=1 — it's to **define what "drift" means** so the same metric scales cleanly to many markets later.

Working definition for this notebook:

- **range** = `max(YES price) − min(YES price)` within the window. Captures the widest swing.
- **std-dev** = `std(YES price)` within the window. Captures volatility around the mean.

A larger live-window range or std-dev than pre-game would be consistent with more new information arriving once the game starts. The directional question — *does drift typically widen or tighten at the bell?* — needs N>>1 markets to answer.

In [36]:
trade_min = trades_df["trade_time"].min()
trade_max = trades_df["trade_time"].max()

# pd.cut needs strictly increasing bin edges. If GAME_START falls outside the
# observed trade range, all trades end up in one window and the cut breaks —
# usually a sign the timezone assumption is wrong (see plot-trades cell).
if GAME_START <= trade_min:
    raise ValueError(
        f"GAME_START ({GAME_START}) is at/before the first trade ({trade_min}). "
        f"All trades would be classified as 'live'. Adjust the timezone assumption."
    )
if GAME_START >= trade_max:
    raise ValueError(
        f"GAME_START ({GAME_START}) is at/after the last trade ({trade_max}). "
        f"All trades would be classified as 'pre-game'. Adjust the timezone assumption."
    )

trades_df["window"] = pd.cut(
    trades_df["trade_time"],
    bins=[
        trade_min - pd.Timedelta(seconds=1),
        GAME_START,
        trade_max + pd.Timedelta(seconds=1),
    ],
    labels=["pre-game", "live"],
)

summary = (
    trades_df.groupby("window", observed=True)
    .agg(
        n_trades=("trade_id", "count"),
        first_trade=("trade_time", "min"),
        last_trade=("trade_time", "max"),
        yes_price_mean=("yes_price_dollars", "mean"),
        yes_price_std=("yes_price_dollars", "std"),
        yes_price_min=("yes_price_dollars", "min"),
        yes_price_max=("yes_price_dollars", "max"),
    )
    .assign(yes_price_range=lambda d: d["yes_price_max"] - d["yes_price_min"])
)
summary

,n_trades,first_trade,last_trade,yes_price_mean,yes_price_std,yes_price_min,yes_price_max,yes_price_range
window,,,,,,,,
pre-game,83,2026-05-10 04:40:56.083379-04:00,2026-05-10 19:19:12.960434-04:00,0.352530,0.007130,0.34,0.36,0.02
live,779,2026-05-10 19:21:20.974128-04:00,2026-05-10 22:24:32.659995-04:00,0.560719,0.216976,0.21,0.99,0.78


## Observable findings

_Fill these in after running the cells above. Discipline (see project memory note on narrative drift): describe **what the data shows**, not a causal story for **why** it moved. Causation needs more than one market and more than one game._

- N trades pre-game: …
- N trades live: …
- YES price range pre-game: $… → $…
- YES price range live: $… → $…
- Std-dev ratio (live / pre-game): …

## What this unlocks

This notebook is the first lap. To get from here to a defensible answer on the headline question:

1. **More markets.** At least one full slate (~15 games × ~5 market types per game = ~75 markets) before pre-game vs. live differences are anything beyond anecdote.
2. **A real `pre-game` boundary.** Game start time should live in a `dim_events` enrichment, not be parsed from the event ticker. That is the next dbt-side piece.
3. **Liquidity context.** The headline question pairs drift with **liquidity**. Need to bring `liquidity_dollars` and `yes_bid_size_fp` / `yes_ask_size_fp` into the comparison once N>1.

Subsequent notebooks (`02_*`, `03_*`) will pick these up.